# step A-1 — 선행 distinct (RQ1 대조군)

**대응 RQ:** RQ1 대조 — step A(clone)의 절벽/바닥/primacy가 선행을 **서로 다른 과제 12개**로 바꿔도 재현되는가.

**무엇을 확인하나**
- step A와 **composition만 다름**(clone→distinct). 나머지(camel·약함·seed20·생성3) 동일.
- distinct에서도 같은 패턴이면 → '데모 오버라이드'가 아니라 **형태 신호** → RQ1 견고.

설계: `docs/stepA-1/plan.md`. (camel-on-Python이라 step A처럼 바닥일 가능성 — 그래도 clone 대비 패턴 비교가 목적.)

> 메모리·시간은 step A와 동일(3B fp16, 100회 ~20–40분). 재개 가능.

In [ ]:
# 환경 설정 — 설치, GPU 확인, 시드 고정
!pip install -q transformers accelerate torch matplotlib pandas
import random, numpy as np, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
SEED=0; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('seed fixed:', SEED)

In [ ]:
# 저장소 클론 및 브랜치 체크아웃
import os
if not os.path.isdir('HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git
%cd HCLT_2026
!git fetch --quiet origin stepA-1/prefix-distinct
!git checkout stepA-1/prefix-distinct
!git pull --quiet origin stepA-1/prefix-distinct
!pip install -e . -q
import sys; sys.path.insert(0, 'src')

In [ ]:
# 조건 설정 — step A와 composition만 다르다 (DISTINCT)
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Composition, InstructionForm, Notation)
MODEL = ModelSpec(name='Qwen/Qwen2.5-Coder-3B-Instruct', family='qwen', dtype='float16')
N_COMPLIANT = [4, 3, 2, 1, 0]
SEEDS = list(range(20))
STEP = 'stepA-1'
def make(n, s):
    return Condition(model=MODEL,
                     preceding=PrecedingCode(n_compliant=n, composition=Composition.DISTINCT),
                     instruction=Instruction(form=InstructionForm.POSITIVE, target_notation=Notation.CAMEL),
                     seed=s)
conditions = [make(n, s) for n in N_COMPLIANT for s in SEEDS]
PREDICTION = ('선행이 서로 다른 과제여도 clone과 유사한 바닥/primacy/잠금이면 데모 오버라이드가 아님. '
              'camel-on-Python이라 준수율은 바닥 예상.')
print(len(conditions), '개 조건')

In [ ]:
# 실행 — 조건별 순차 생성 + 즉시 저장(재개 가능). 원문 자동 저장.
from harness import run, ResultRecord, save_result, result_path
from harness.model import load_model
handle = load_model(MODEL)
print('layers:', handle.num_layers, '| GQA:', handle.gqa_info())
new = skipped = 0
for c in conditions:
    if result_path(c, step=STEP).exists():
        skipped += 1; continue
    out = run(c, handle=handle)
    save_result(ResultRecord(condition=out.condition, metrics=out.metrics,
                             step=STEP, rq='RQ1', prediction=PREDICTION))
    new += 1
    if new % 10 == 0: print(f'생성 {new} / 건너뜀 {skipped} / 총 {len(conditions)}')
print(f'완료: 새로 {new}, 건너뜀 {skipped}, 총 {len(conditions)}')

In [ ]:
# 결과 로드 — results/stepA-1/
from harness import result_path
from harness.results import load_result
records = [load_result(result_path(c, step=STEP)) for c in conditions]
print('로드:', len(records), '건 → results/'+STEP+'/')

In [ ]:
# 요약 — clone(step A) vs distinct(step A-1) 준수율 겹쳐 그리기
import pandas as pd, matplotlib.pyplot as plt, glob, json
def rate_by_n(step):
    rows=[]
    for f in glob.glob(f'results/{step}/*.json'):
        d=json.load(open(f)); e=d['metrics']['extra']
        rows.append({'n':d['condition']['preceding']['n_compliant'],'c':e['first_compliant']})
    if not rows: return None
    return pd.DataFrame(rows).groupby('n')['c'].mean().reindex(N_COMPLIANT)
d1 = rate_by_n('stepA-1'); d0 = rate_by_n('stepA')
print('distinct(A-1) 준수율:'); print(d1.round(3))
plt.figure(figsize=(5,3))
plt.plot([str(n) for n in N_COMPLIANT], d1.values, marker='o', label='distinct (A-1)')
if d0 is not None:
    plt.plot([str(n) for n in N_COMPLIANT], d0.values, marker='s', label='clone (A)')
plt.xlabel('n_compliant (4->0)'); plt.ylabel('준수율(첫함수)'); plt.ylim(-0.02,1.02)
plt.legend(); plt.grid(alpha=.3); plt.title('RQ1: clone vs distinct'); plt.show()